# Phase 3 · Questions
**DressMe · Encoder Validation Lab**

**Goal of this phase:** one **frozen** file of questions that every model will answer, built with the same rules as the app's step 3b calibration: English **words** questions and **photo** questions, 1,000 per track per split (validation, test), 30% of them with no right answer, with integrity and leakage checks. **No model is used in this phase.** French and Arabic come later, as a separate add-on.

**What a question is:** a fake wardrobe of 30 of the user's pieces, a query (the words *"black jeans"*, or a photo of black jeans), and the list of pieces that are the right answer (1 to 3 of them, or none).

**Before you start (right-hand panel):**
- **Accelerator:** None. **Internet:** On.
- **Inputs (Add Input):** your **Phase 2 notebook's output** (Your Work > Notebooks > your Phase 2 notebook, the version saved with Save & Run All), and **paramaggarwal/fashion-product-images-dataset** (full).

**How to run:** one cell at a time (Shift + Enter). Read the card, predict, run, compare.

### Step 3.0: Settings and inputs
- **What it does:** writes down every rule of the question generator in one place (seed, wardrobe size 30, 1,000 questions per track per split, 30% without a right answer, at least 3 pieces for a query to be allowed, 1 to 3 right pieces), then searches `/kaggle/input` for the Phase 2 lists (`items_val.csv`, `items_test.csv`) and for Kaggle's full `images` folder. Prints the library versions.
- **Why:** the rules are copied from the app's `calibrate_search.py`, so this exam asks the same kind of questions the app was calibrated on. Writing them in one place, before anything runs, is what makes them checkable.
- **Effect:** fixes the seeds. Nothing is saved or downloaded.
- **Predict:** Kaggle's dataset has two `images` folders (the hidden duplicate from Phase 2): which one should be chosen?
- **You should see:** a table with the two lists and the images folder (the **outer** one, `…/fashion-dataset/images`, not `…/fashion-dataset/fashion-dataset/images`), a line `Sample image: … px -> full resolution`, the note that 2 image folders were found, then the versions.
- **If it looks wrong:** `items_val.csv not found` → Add Input > Your Work > Notebooks > your Phase 2 notebook. If it isn't listed there, its version wasn't saved with **Save & Run All**: go back and do that first. A message about small images → attach the full dataset.
- **Teacher angle:** "Are these the same rules the app uses?" Yes: each rule is a named setting copied from the app's calibration script, printed at the top.

In [ ]:
# ---- Step 3.0: settings (the generator's rules), and find the inputs ----
import os, sys, json, random, platform, hashlib, datetime
import numpy as np
import pandas as pd
import PIL
from PIL import Image

SEED                = 0
INPUT_ROOT          = "/kaggle/input"
OUT_DIR             = "/kaggle/working"
SPLITS, SPLIT_NO    = ["val", "test"], {"val": 0, "test": 1}
TRACKS, TRACK_NO    = ["words", "photo"], {"words": 0, "photo": 1}
WARDROBE_SIZE       = 30      # pieces in each fake wardrobe
QUESTIONS_PER_TRACK = 1000    # per track, per split -> 4,000 questions in total
NO_ANSWER_SHARE     = 0.3     # share of questions where the right piece is NOT in the wardrobe
TYPE_FORM_SHARE     = 0.5     # words: "jeans" (type only) vs "black jeans" (colour + type)
MIN_PIECES          = 3       # a type or a (colour, type) pair needs >= 3 items in the split
K_CHOICES           = (1, 2, 3)   # number of right pieces in a question that has an answer
EXPECTED_ITEMS      = {"val": 3402, "test": 3548}   # from Phase 2
SMALL_IMAGE_PX      = 200
REPORT              = {}

random.seed(SEED)
np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.isdir(INPUT_ROOT):
    raise RuntimeError(f"No {INPUT_ROOT} folder: run this notebook on Kaggle with the two inputs attached.")

found = {"items_val.csv": [], "items_test.csv": []}
images_found = []
for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
    for name in found:
        if name in filenames:
            found[name].append(os.path.join(dirpath, name))
    if "images" in dirnames:
        images_found.append(os.path.join(dirpath, "images"))
    dirnames[:] = [d for d in dirnames if d not in ("images", "styles")]   # don't walk 44,000 files

for name, paths in found.items():
    if not paths:
        raise RuntimeError(f"{name} not found: Add Input > Your Work > Notebooks > your Phase 2 notebook "
                           "(the version saved with Save & Run All), then rerun this cell.")
if not images_found:
    raise RuntimeError("Kaggle's images not found: Add Input > 'fashion-product-images-dataset' "
                       "(by paramaggarwal, the FULL one), then rerun.")

def sample_size(folder):
    first = next((n for n in sorted(os.listdir(folder)) if n.endswith(".jpg")), None)
    if first is None:
        return (0, 0)
    with Image.open(os.path.join(folder, first)) as im:
        return im.size

sizes = {d: sample_size(d) for d in images_found}
IMAGES_DIR = max(sizes, key=lambda d: (sizes[d][0], -len(d)))   # largest pictures; if equal, the outer folder
SAMPLE_W, SAMPLE_H = sizes[IMAGES_DIR]
if SAMPLE_W < SMALL_IMAGE_PX:
    raise RuntimeError(f"The images are only {SAMPLE_W} x {SAMPLE_H} px: attach the FULL fashion-product-images-dataset.")
ITEMS_CSV = {"val": found["items_val.csv"][0], "test": found["items_test.csv"][0]}

print(pd.DataFrame([("Validation items (Phase 2)", ITEMS_CSV["val"]),
                    ("Test items (Phase 2)",       ITEMS_CSV["test"]),
                    ("Kaggle's full images",       IMAGES_DIR),
                    ("Output folder",              OUT_DIR)], columns=["Input", "Path"]).to_string(index=False))
print(f"\nSample image: {SAMPLE_W} x {SAMPLE_H} px -> full resolution")
if len(images_found) > 1:
    print(f"(Found {len(images_found)} image folders, as in Phase 2; using the outer one.)")
print(f"\nRules: wardrobe {WARDROBE_SIZE} · {QUESTIONS_PER_TRACK:,} questions per track per split · "
      f"no-answer share {NO_ANSWER_SHARE} · minimum {MIN_PIECES} pieces · k in {K_CHOICES} · seed {SEED}\n")
VERSIONS = {"python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "pillow": PIL.__version__}
for name, ver in VERSIONS.items():
    print(f"{name:<8} {ver}")

### Step 3.1: Vocabulary
- **What it does:** loads the validation and test items from Phase 2 and **rebuilds every picture path** from the Kaggle id (`<images folder>/<kaggle_id>.jpg`) instead of trusting the saved one, then checks every file exists. Then it builds the **vocabulary** of each split: which types ("jeans") and which colour + type pairs ("black jeans") have **at least 3 items** and may be asked. The rest are listed with their counts (the full list is saved to `phase3_left_out.csv`). Last, it shows how a type is written in a query: `casual-shoes` → `casual shoes`, but `t-shirt` stays `t-shirt` (that's how people type it).
- **Why:** a saved path can go stale: Phase 2's paths point to the folder name Kaggle used that day, and Kaggle has already changed its folder layout once. Rebuilding from the id is always right. The 3-item minimum comes from the app's rules: a query needs enough pieces to have 1 to 3 right answers and still be a real choice. This is where Phase 2's rare types are decided: types with fewer than 3 items in a split are not asked in that split.
- **Effect:** two tables in memory, one per split. Writes `phase3_left_out.csv`. Takes a few seconds.
- **Predict:** out of roughly 50 types per split, how many will be left out: about 5, 20 or 40?
- **You should see:** a table per split: items (3,402 and 3,548, `PASS`), pictures found (`PASS`), then the number of types and pairs and how many are allowed. Roughly **45 of 50 types** allowed, and only **a few hundred of the many colour + type pairs** (many pairs, like "purple belt", have only 1 or 2 items). Then the left-out types (expect the rare ones from Phase 2: blazer, jumpsuit, gloves…), the 20 largest left-out pairs, and 10 examples of how types are written.
- **If it looks wrong:** a FAIL on items → the wrong Phase 2 version is attached. A FAIL on pictures → the images folder is wrong (check step 3.0). Journal the left-out types: they answer "what did you do with rare types?"
- **Teacher angle:** "What did you do with types that have only one or two items?" They're kept in the wardrobes, but never asked as a query: a query needs at least 3 pieces, the app's own rule.

In [ ]:
# ---- Step 3.1: load the items, rebuild the paths, and build the vocabulary ----
NEEDED = ["item_id", "kaggle_id", "split", "type", "colour", "image_group"]

def typed(t):
    """How a type is written in a query: hyphens become spaces, except 't-shirt'."""
    return t if t == "t-shirt" else t.replace("-", " ")

def allowed_queries(df, min_pieces=MIN_PIECES):
    """Types and (colour, type) pairs with at least `min_pieces` items, sorted (fixed order = reproducible)."""
    t_counts = df["type"].value_counts()
    p_counts = df.groupby(["colour", "type"]).size()
    return {"types": sorted(t for t, n in t_counts.items() if n >= min_pieces),
            "pairs": sorted(p for p, n in p_counts.items() if n >= min_pieces),
            "type_counts": t_counts, "pair_counts": p_counts}

image_files = set(os.listdir(IMAGES_DIR))
ITEMS, ALLOWED, rows, left_out = {}, {}, [], []
for s in SPLITS:
    df = pd.read_csv(ITEMS_CSV[s], dtype=str, keep_default_na=False)
    missing = [c for c in NEEDED if c not in df.columns]
    if missing:
        raise RuntimeError(f"items_{s}.csv lacks {missing}: attach the output of the Phase 2 notebook.")
    for c in NEEDED:
        df[c] = df[c].astype(str).str.strip()
    df["kaggle_id"] = df["kaggle_id"].astype(int)
    df["image_path"] = [os.path.join(IMAGES_DIR, f"{k}.jpg") for k in df["kaggle_id"]]   # rebuilt, not trusted
    df["group_key"] = np.where(df["image_group"] != "", df["image_group"], df["item_id"])  # no group = its own group
    df = df.sort_values("item_id").reset_index(drop=True)
    ITEMS[s], ALLOWED[s] = df, allowed_queries(df)
    a = ALLOWED[s]
    n_pics = sum(f"{k}.jpg" in image_files for k in df["kaggle_id"])
    wrong_split = int((df["split"] != s).sum())
    rows.append((s, f"{len(df):,} / {EXPECTED_ITEMS[s]:,}", "PASS" if len(df) == EXPECTED_ITEMS[s] and wrong_split == 0 else "FAIL",
                 f"{n_pics:,}", "PASS" if n_pics == len(df) else "FAIL",
                 f"{len(a['types'])} of {len(a['type_counts'])}", f"{len(a['pairs'])} of {len(a['pair_counts'])}"))
    for t, n in a["type_counts"].items():
        if n < MIN_PIECES:
            left_out.append({"split": s, "kind": "type", "type": t, "colour": "", "items": int(n)})
    for (c, t), n in a["pair_counts"].items():
        if n < MIN_PIECES:
            left_out.append({"split": s, "kind": "pair", "type": t, "colour": c, "items": int(n)})

print(pd.DataFrame(rows, columns=["Split", "Items / expected", "Check", "Pictures found", "Check",
                                  "Types allowed", "Pairs allowed"]).to_string(index=False))

left = pd.DataFrame(left_out)
left.to_csv(os.path.join(OUT_DIR, "phase3_left_out.csv"), index=False)
for s in SPLITS:
    lt = left[(left["split"] == s) & (left["kind"] == "type")].sort_values("items")
    lp = left[(left["split"] == s) & (left["kind"] == "pair")].sort_values(["items", "type"], ascending=[False, True])
    print(f"\n{s}: types left out (< {MIN_PIECES} items): " +
          (", ".join(f"{r.type} ({r.items})" for r in lt.itertuples()) or "none"))
    print(f"{s}: {len(lp)} pairs left out; the 20 largest: " +
          ", ".join(f"{r.colour} {r.type} ({r.items})" for r in lp.head(20).itertuples()))
print(f"\nFull list saved: {os.path.join(OUT_DIR, 'phase3_left_out.csv')}")

examples = sorted(t for t in ALLOWED["val"]["type_counts"].index if "-" in t and t != "t-shirt")[:9] + ["t-shirt"]
print("\nHow types are written in a query:")
for t in examples:
    print(f"  {t:<16} -> '{typed(t)}'")

REPORT["3.1_vocabulary"] = {s: {"items": len(ITEMS[s]), "types_allowed": len(ALLOWED[s]["types"]),
                                "types_total": len(ALLOWED[s]["type_counts"]),
                                "pairs_allowed": len(ALLOWED[s]["pairs"]),
                                "pairs_total": len(ALLOWED[s]["pair_counts"])} for s in SPLITS}
REPORT["3.1_types_left_out"] = {s: {r.type: r.items for r in left[(left["split"] == s) & (left["kind"] == "type")].itertuples()}
                                for s in SPLITS}

### Step 3.2a: The question builder
- **What it does:** defines the two recipes (functions) that build one question each, following the app's rules. **Nothing runs yet**: this cell only writes the recipes down; 3.2b tests them.<br>**Words question:** flip a coin weighted 30/70 for "no right answer"; flip a fair coin for the form ("jeans" or "black jeans"); pick the query uniformly among the allowed types or pairs; if the question has an answer, pick k = 1, 2 or 3 right pieces among the items that match; fill the rest of the 30 places with items that **don't** match; shuffle the wardrobe.<br>**Photo question:** same, but the query is always a colour + type pair, and the query is **one item's photo**. That item, and every item sharing its picture group, is kept **out** of the wardrobe; the right pieces are the *other* items of the pair.<br>Every random choice comes from a **random stream** (a generator started from a fixed seed), one stream per track and split, so the 4,000 questions come out identical on every run.
- **Why:** the generator is the heart of the exam: if it's wrong, every score in Phase 5 is wrong. Writing it as two short recipes makes it testable by eye (3.2b) and by machine (3.5).
- **Effect:** nothing runs and nothing is saved; two functions are defined.
- **Predict:** in a no-answer question, can a blue pair of jeans still be in the wardrobe when the query is "black jeans"?
- **You should see:** two lines: `Question builders ready: words_question, photo_question`, plus the rules they use.
- **If it looks wrong:** an error here is a typing problem in the cell: copy it again exactly. Nothing else can go wrong yet.
- **Teacher angle:** "How exactly are your questions built?" These two recipes, with every random choice from a fixed seed.

In [ ]:
# ---- Step 3.2a: the two question builders (defined here, tested in 3.2b) ----
def make_pool(df):
    """The split as plain arrays, in a fixed order (sorted by item_id)."""
    return {"id": df["item_id"].to_numpy(), "type": df["type"].to_numpy(),
            "colour": df["colour"].to_numpy(), "group": df["group_key"].to_numpy()}

def pick_right_and_fill(rng, absent, matching, others, size):
    """k right pieces (none if absent), the rest from non-matching items, then shuffled."""
    right = []
    if not absent:
        k = min(int(rng.choice(K_CHOICES)), len(matching))          # k capped by how many exist
        right = [str(x) for x in rng.choice(matching, size=k, replace=False)]
    fillers = [str(x) for x in rng.choice(others, size=size - len(right), replace=False)]
    wardrobe = [str(x) for x in rng.permutation(right + fillers)]  # shuffle: the answer's position is random
    return wardrobe, sorted(right)

def words_question(rng, pool, allowed, size=WARDROBE_SIZE):
    absent = bool(rng.random() < NO_ANSWER_SHARE)
    if rng.random() < TYPE_FORM_SHARE:
        form, qcolour = "type", None
        qtype = allowed["types"][rng.integers(len(allowed["types"]))]
        match = pool["type"] == qtype
        text = typed(qtype)
    else:
        form = "colour+type"
        qcolour, qtype = allowed["pairs"][rng.integers(len(allowed["pairs"]))]
        match = (pool["type"] == qtype) & (pool["colour"] == qcolour)
        text = f"{qcolour} {typed(qtype)}"
    wardrobe, right = pick_right_and_fill(rng, absent, pool["id"][match], pool["id"][~match], size)
    return {"form": form, "query_text": text, "query_type": qtype, "query_colour": qcolour,
            "absent": absent, "wardrobe": wardrobe, "right": right}

def usable_query_items(pool, in_pair):
    """Items of the pair that leave at least one other pair item outside their own picture group."""
    idx = np.flatnonzero(in_pair)
    groups = pool["group"][idx]
    return [i for i, g in zip(idx, groups) if (groups != g).any()]

def photo_pairs(pool, pairs):
    """Allowed pairs that can be asked as a photo (some item leaves a right piece after the group rule)."""
    return [p for p in pairs if usable_query_items(pool, (pool["colour"] == p[0]) & (pool["type"] == p[1]))]

def photo_question(rng, pool, pairs, size=WARDROBE_SIZE):
    absent = bool(rng.random() < NO_ANSWER_SHARE)
    qcolour, qtype = pairs[rng.integers(len(pairs))]
    in_pair = (pool["type"] == qtype) & (pool["colour"] == qcolour)
    usable = usable_query_items(pool, in_pair)
    q = usable[rng.integers(len(usable))]                          # the query photo
    same_group = pool["group"] == pool["group"][q]                 # the query + identical pictures: kept out
    wardrobe, right = pick_right_and_fill(rng, absent, pool["id"][in_pair & ~same_group],
                                          pool["id"][~in_pair & ~same_group], size)
    return {"form": "colour+type", "query_item_id": str(pool["id"][q]), "query_type": qtype,
            "query_colour": qcolour, "absent": absent, "wardrobe": wardrobe, "right": right}

print("Question builders ready: words_question, photo_question")
print(f"Rules: no-answer {NO_ANSWER_SHARE:.0%} · type form {TYPE_FORM_SHARE:.0%} · k in {K_CHOICES} · "
      f"wardrobe {WARDROBE_SIZE} · query needs >= {MIN_PIECES} items")

### Step 3.2b: Test the builder on a toy pool
- **What it does:** makes up a tiny pool of **12 items** (black t-shirts, a white t-shirt, blue jeans, black jeans, black and brown shoes; two of the blue jeans share one picture group), uses a small wardrobe of **6**, and prints one question **with** an answer and one **without** for each track, each as a small table where the right pieces are marked.
- **Why:** with 3,400 items and wardrobes of 30, a mistake hides in the crowd. With 12 items you can check every line by eye in a minute: are the marked pieces really the right answer? Is nothing else a right answer? Is the query photo absent from its own wardrobe, and is its twin (same picture group) absent too? Testing on a case small enough to check by hand is the same idea as Phase 5's toy test: measure something whose answer you already know.
- **Effect:** nothing saved; prints four small tables.
- **Predict:** in the photo question with an answer, will the query's twin (same picture group) appear in the wardrobe?
- **You should see:** four blocks. Words with an answer: the marked pieces have the query's type (and colour, for the colour + type form), and no unmarked piece does. Words without an answer: no piece is marked, and none matches. Photo: the query item is listed above its wardrobe and **isn't in it**; if the query is a blue jeans from the shared group, its twin isn't there either.
- **If it looks wrong:** a marked piece that doesn't match the query, an unmarked piece that does, or the query photo inside its own wardrobe → the builder is wrong. Stop and bring the four tables to me.
- **Teacher angle:** "How do you know your question generator is right?" I first ran it on 12 items, where every answer can be checked by hand, and then checked all 4,000 by machine (step 3.5).

In [ ]:
# ---- Step 3.2b: run the builders on a 12-item toy pool, wardrobe of 6 ----
toy = pd.DataFrame([
    ("toy01", "t-shirt", "black", "g01"), ("toy02", "t-shirt", "black", "g02"), ("toy03", "t-shirt", "black", "g03"),
    ("toy04", "t-shirt", "white", "g04"), ("toy05", "jeans", "blue", "gJJ"), ("toy06", "jeans", "blue", "gJJ"),
    ("toy07", "jeans", "blue", "g07"),    ("toy08", "jeans", "black", "g08"), ("toy09", "casual-shoes", "black", "g09"),
    ("toy10", "casual-shoes", "black", "g10"), ("toy11", "casual-shoes", "black", "g11"),
    ("toy12", "casual-shoes", "brown", "g12"),
], columns=["item_id", "type", "colour", "group_key"])
toy_pool, toy_allowed = make_pool(toy), allowed_queries(toy)
toy_photo_pairs = photo_pairs(toy_pool, toy_allowed["pairs"])
print(f"Toy pool: {len(toy)} items · allowed types {toy_allowed['types']} · allowed pairs {toy_allowed['pairs']}\n")

def show(title, q, df):
    print(f"--- {title} ---")
    if "query_text" in q:
        print(f"query: '{q['query_text']}'  (form: {q['form']}) · absent: {q['absent']} · right: {q['right'] or 'none'}")
    else:
        qi = df.set_index("item_id").loc[q["query_item_id"]]
        print(f"query photo: {q['query_item_id']} ({qi['colour']} {qi['type']}, group {qi['group_key']}) · "
              f"absent: {q['absent']} · right: {q['right'] or 'none'}")
    w = df.set_index("item_id").loc[q["wardrobe"], ["type", "colour", "group_key"]].reset_index()
    w["right?"] = np.where(w["item_id"].isin(q["right"]), "RIGHT", "")
    print(w.to_string(index=False), "\n")

toy_rng = np.random.default_rng([SEED, 99])
for track, build in [("words", lambda: words_question(toy_rng, toy_pool, toy_allowed, size=6)),
                     ("photo", lambda: photo_question(toy_rng, toy_pool, toy_photo_pairs, size=6))]:
    shown = set()
    for _ in range(500):
        q = build()
        if q["absent"] not in shown:
            show(f"{track} question {'WITHOUT' if q['absent'] else 'WITH'} an answer", q, toy)
            shown.add(q["absent"])
        if len(shown) == 2:
            break

### Step 3.3: Words questions
- **What it does:** builds **1,000 words questions per split** (validation, test) with the builder from 3.2a, each split with its own fixed random stream (`[seed 0, track 0, split]`). Prints, per split: how many have an answer and how many don't, the share of each form ("jeans" vs "black jeans"), how many right pieces the questions have (k = 1, 2, 3), and the 10 most asked queries.<br>**Ground truth** means the answer key: for each question, the list of pieces that are right, decided by the labels, not by any model. There can be **several** right pieces, because a real wardrobe can hold two pairs of black jeans: if the user types "black jeans", both are right, and a model finding either one has done its job.<br>**Why 30% without an answer:** in real use, people search for things they don't own ("do I have a red dress?"). A good search must then say honestly "no sure match" instead of showing the closest wrong piece. If every question had an answer, a model that always shows *something* would look perfect. The no-answer questions measure that honest "no sure match" (Phase 5 scores it with each model's cut).
- **Why:** this is the main exam: Phase 5's winner is chosen on these words questions (test split). Checking the mix now (forms, k, the most asked queries) proves the exam has the shape we intended.
- **Effect:** 2,000 questions in memory (a few MB). Nothing saved yet (step 3.6 saves everything).
- **Predict:** out of 1,000, how many questions will have no right answer: exactly 300, or about 300?
- **You should see:** for each split: about **700 with an answer and 300 without** (between roughly 270 and 330: it's drawn per question, like flipping 1,000 weighted coins); about half of each form; k = 1, 2, 3 in about equal thirds; the most asked queries each appear only around 10–25 times, because queries are picked uniformly among many allowed types and pairs.
- **If it looks wrong:** a no-answer share far from 30% (say 10% or 60%), or one form far from half → a setting in 3.0 was changed. A single query asked hundreds of times → the query pick is not uniform. Journal it.
- **Teacher angle:** "What is your ground truth, and why can a question have several right answers?" The labels decide it; two black jeans are both right for "black jeans".

In [ ]:
# ---- Step 3.3: 1,000 words questions per split ----
POOLS = {s: make_pool(ITEMS[s]) for s in SPLITS}
QUESTIONS = []

def track_stats(qs):
    """Counts for one track and split: answer / no answer, forms, k, top queries."""
    present = [q for q in qs if not q["absent"]]
    k = pd.Series([len(q["right"]) for q in present]).value_counts().reindex([1, 2, 3], fill_value=0)
    return {"questions": len(qs), "with_answer": len(present), "no_answer": len(qs) - len(present),
            "type_form": sum(q["form"] == "type" for q in qs),
            "colour_type_form": sum(q["form"] == "colour+type" for q in qs),
            "k1": int(k[1]), "k2": int(k[2]), "k3": int(k[3])}

def print_stats(stats):
    t = pd.DataFrame(stats).T
    t["no-answer share"] = (t["no_answer"] / t["questions"]).map(lambda x: f"{x:.1%}")
    print(t[["questions", "with_answer", "no_answer", "no-answer share", "type_form", "colour_type_form",
             "k1", "k2", "k3"]].to_string())

words_stats = {}
for s in SPLITS:
    rng = np.random.default_rng([SEED, TRACK_NO["words"], SPLIT_NO[s]])
    qs = []
    for i in range(QUESTIONS_PER_TRACK):
        q = words_question(rng, POOLS[s], ALLOWED[s])
        qs.append({"qid": f"{s}-words-{i + 1:04d}", "split": s, "track": "words", **q})
    QUESTIONS.extend(qs)
    words_stats[s] = track_stats(qs)

print("Words questions (k = number of right pieces, among questions that have an answer):")
print_stats(words_stats)
for s in SPLITS:
    top = pd.Series([q["query_text"] for q in QUESTIONS if q["split"] == s and q["track"] == "words"]).value_counts().head(10)
    print(f"\n{s}: the 10 most asked queries: " + ", ".join(f"'{t}' ({n})" for t, n in top.items()))
REPORT["3.3_words"] = words_stats

### Step 3.4: Photo questions
- **What it does:** builds **1,000 photo questions per split**, each split with its own stream (`[seed 0, track 1, split]`). The query is the photo of one item; the right pieces are the *other* items of the same colour + type. Prints the same counts as 3.3, then a **leakage check** on every photo question: the query item is never in its own wardrobe, and no item sharing its picture group is either.<br>**Leakage, the identical-picture example:** some catalogue products share the exact same photo. If the query photo, or an identical copy, sat in the wardrobe, every model would find it with a perfect score, because comparing a picture with itself always wins. The test would measure "can you find the same file", not "can you find the same garment". That's why the query item and its whole picture group are kept out.
- **Why:** photo search is the second job of the encoder (and rule line 5 disqualifies a model that is clearly worse on photos). The leakage check proves that a good photo score will mean the model recognises the garment, not the file.
- **Effect:** 2,000 more questions in memory (4,000 in total). Nothing saved yet.
- **Predict:** will any photo question have its own query photo in the wardrobe?
- **You should see:** a line saying how many allowed pairs can be asked as a photo (all or nearly all), then the same table as 3.3 (about 700 / 300; all questions in the colour + type form; k = 3 a little rarer than 1 and 2, because a pair of exactly 3 items leaves only 2 others once the query is taken out), the 10 most asked pairs, and two leakage lines: **0 and 0, both PASS**.
- **If it looks wrong:** any leakage FAIL is serious: stop and bring the output to me, because the scores would be inflated. Many pairs "not usable as a photo" would mean most items of those pairs share one picture group: note the number.
- **Teacher angle:** "Could a model win the photo track just by matching the identical picture?" No: the query and every picture in its group are excluded from its wardrobe, and the check proves it for all 2,000 photo questions.

In [ ]:
# ---- Step 3.4: 1,000 photo questions per split, then the leakage check ----
PHOTO_PAIRS, photo_stats = {}, {}
for s in SPLITS:
    PHOTO_PAIRS[s] = photo_pairs(POOLS[s], ALLOWED[s]["pairs"])
    print(f"{s}: {len(PHOTO_PAIRS[s])} of {len(ALLOWED[s]['pairs'])} allowed pairs can be asked as a photo")
    rng = np.random.default_rng([SEED, TRACK_NO["photo"], SPLIT_NO[s]])
    qs = []
    for i in range(QUESTIONS_PER_TRACK):
        q = photo_question(rng, POOLS[s], PHOTO_PAIRS[s])
        qs.append({"qid": f"{s}-photo-{i + 1:04d}", "split": s, "track": "photo", **q})
    QUESTIONS.extend(qs)
    photo_stats[s] = track_stats(qs)

print("\nPhoto questions:")
print_stats(photo_stats)
for s in SPLITS:
    top = pd.Series([f"{q['query_colour']} {q['query_type']}" for q in QUESTIONS
                     if q["split"] == s and q["track"] == "photo"]).value_counts().head(10)
    print(f"\n{s}: the 10 most asked pairs: " + ", ".join(f"{t} ({n})" for t, n in top.items()))

GROUP_OF = {s: dict(zip(ITEMS[s]["item_id"], ITEMS[s]["group_key"])) for s in SPLITS}
photo_qs = [q for q in QUESTIONS if q["track"] == "photo"]
self_in = sum(q["query_item_id"] in q["wardrobe"] for q in photo_qs)
group_in = sum(GROUP_OF[q["split"]][q["query_item_id"]] in {GROUP_OF[q["split"]][w] for w in q["wardrobe"]}
               for q in photo_qs)
print(f"\nLeakage check over {len(photo_qs):,} photo questions:")
print(f"  query item inside its own wardrobe:        {self_in}   {'PASS' if self_in == 0 else 'FAIL'}")
print(f"  query's picture group inside its wardrobe: {group_in}   {'PASS' if group_in == 0 else 'FAIL'}")

REPORT["3.4_photo"] = photo_stats
REPORT["3.4_leakage"] = {"query_in_wardrobe": self_in, "query_group_in_wardrobe": group_in,
                         "photo_pairs": {s: len(PHOTO_PAIRS[s]) for s in SPLITS}}

### Step 3.5: Integrity checks
- **What it does:** checks, by machine, **all 4,000 questions** against seven rules, and prints how many break each one (PASS when 0): (1) the wardrobe has exactly 30 pieces; (2) no piece appears twice in a wardrobe; (3) every piece comes from the question's own split; (4) every right piece is inside the wardrobe; (5) no-answer questions have 0 right pieces, the others 1 to 3; (6) every right piece really matches the query's labels; (7) no *wrong* piece matches them (otherwise a model finding it would be marked wrong for a right answer).
- **Why:** 3.2b checked a few questions by eye; this checks every question by rule. Together they're the proof that the exam is correct before any model sees it. A broken question can't be fixed after the scores exist without suspicion.
- **Effect:** nothing saved; prints a seven-line table.
- **Predict:** out of 4,000 questions × 7 rules, how many failures do you expect?
- **You should see:** a table of seven checks, each with **0** failures and `PASS`, then `All integrity checks passed`.
- **If it looks wrong:** any non-zero line → the questions are not safe to freeze. Don't run 3.6; bring me the table. (The most likely cause is an edited cell in 3.2a.)
- **Teacher angle:** "How do you know every one of your 4,000 questions is correct?" Seven rules were checked by machine on every question, all with zero failures.

In [ ]:
# ---- Step 3.5: seven integrity rules, checked on all 4,000 questions ----
LABELS = {s: ITEMS[s].set_index("item_id")[["type", "colour"]].to_dict("index") for s in SPLITS}

def matches(q, item_id):
    lab = LABELS[q["split"]][item_id]
    if q["form"] == "type":
        return lab["type"] == q["query_type"]
    return lab["type"] == q["query_type"] and lab["colour"] == q["query_colour"]

checks = {
    "1. wardrobe has exactly 30 pieces":            lambda q: len(q["wardrobe"]) == WARDROBE_SIZE,
    "2. no piece repeated in a wardrobe":            lambda q: len(set(q["wardrobe"])) == len(q["wardrobe"]),
    "3. every piece is from the question's split":   lambda q: all(w in LABELS[q["split"]] for w in q["wardrobe"]),
    "4. right pieces are inside the wardrobe":       lambda q: set(q["right"]) <= set(q["wardrobe"]),
    "5. no-answer: 0 right · others: 1 to 3":        lambda q: len(q["right"]) == 0 if q["absent"] else 1 <= len(q["right"]) <= 3,
    "6. every right piece matches the query":        lambda q: all(matches(q, r) for r in q["right"]),
    "7. no wrong piece matches the query":           lambda q: not any(matches(q, w) for w in q["wardrobe"] if w not in q["right"]),
}
rows, integrity = [], {}
for name, rule in checks.items():
    fails = sum(not rule(q) for q in QUESTIONS)
    integrity[name] = fails
    rows.append((name, f"{fails:,}", "PASS" if fails == 0 else "FAIL"))
print(f"Checked {len(QUESTIONS):,} questions:\n")
print(pd.DataFrame(rows, columns=["Rule", "Failures", "Check"]).to_string(index=False))
ALL_OK = all(v == 0 for v in integrity.values())
print("\nAll integrity checks passed." if ALL_OK else "\nSome checks FAILED: do not freeze; bring this table to your helper.")
REPORT["3.5_integrity"] = integrity

### Step 3.6: Freeze
- **What it does:** writes all 4,000 questions to **`questions.json`**: a `meta` part (seed, rules, counts, creation time, versions) and a `questions` part (for each: its id like `val-words-0001`, split, track, form, the query text or the query item, the query's type and colour, whether it has no answer, the 30 wardrobe ids, and the right ids). Query texts are stored **without** "a photo of": Phase 4 adds it when encoding. Then it computes the file's **SHA-256 checksum** and saves it to `questions.sha256.txt`.<br>**Freezing** means the exam is written once and never changed: every model, in Phase 4 and 5, answers this exact file. If each model got freshly drawn questions, one could simply be luckier.<br>**The checksum** is a fingerprint of the file: a 64-character code computed from every byte. Change one character anywhere and the fingerprint changes completely. Recording it now, before any score exists, lets anyone check later that the exam wasn't adjusted after seeing the results. The file contains its creation time, so a rerun gives a *different* file fingerprint; that's why a second fingerprint is printed, computed from the **questions only**: it stays identical on every run and proves the questions themselves are reproducible.
- **Why:** Phase 4 encodes these questions and Phase 5 scores them; the checksum ties both to this exact file. It's the same idea as committing the decision rule before the scores.
- **Effect:** writes `questions.json` (a few MB) and `questions.sha256.txt` to the output folder. Refuses to write if a 3.5 check failed.
- **Predict:** if you rerun this cell, will the file's fingerprint stay the same? And the questions-only fingerprint?
- **You should see:** the file's path and size (a few MB), its **SHA-256** (64 letters and digits), the **questions-only fingerprint**, and the counts per track and split (1,000 each). Write both fingerprints in your journal.
- **If it looks wrong:** `Integrity checks failed` → go back to 3.5. Rerunning gives a new file fingerprint (new creation time) but the **same** questions-only fingerprint; if that one changes between runs, something isn't seeded: bring me both values.
- **Teacher angle:** "How can we be sure the exam wasn't changed after you saw the scores?" Its SHA-256 fingerprint was recorded before any model was run.

In [ ]:
# ---- Step 3.6: freeze the questions into questions.json, with its fingerprint ----
if not ALL_OK:
    raise RuntimeError("Integrity checks failed in step 3.5: the questions must not be frozen. Fix 3.5 first.")

counts = {s: {t: sum(q["split"] == s and q["track"] == t for q in QUESTIONS) for t in TRACKS} for s in SPLITS}
meta = {
    "phase": 3, "seed": SEED,
    "rules": {"wardrobe_size": WARDROBE_SIZE, "questions_per_track_per_split": QUESTIONS_PER_TRACK,
              "no_answer_share": NO_ANSWER_SHARE, "type_form_share": TYPE_FORM_SHARE,
              "min_pieces": MIN_PIECES, "k_choices": list(K_CHOICES),
              "random_streams": "numpy default_rng([seed, track_no, split_no]); words=0, photo=1; val=0, test=1",
              "query_text": "stored without 'a photo of' (added when encoding)",
              "typed": "hyphens -> spaces, except 't-shirt'"},
    "counts": counts,
    "created": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "inputs": {s: os.path.basename(ITEMS_CSV[s]) for s in SPLITS},
    "versions": VERSIONS,
}
QUESTIONS_JSON = os.path.join(OUT_DIR, "questions.json")
with open(QUESTIONS_JSON, "w") as f:
    json.dump({"meta": meta, "questions": QUESTIONS}, f, separators=(",", ":"))

with open(QUESTIONS_JSON, "rb") as f:
    FILE_SHA = hashlib.sha256(f.read()).hexdigest()
CONTENT_SHA = hashlib.sha256(json.dumps(QUESTIONS, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
SHA_TXT = os.path.join(OUT_DIR, "questions.sha256.txt")
with open(SHA_TXT, "w") as f:
    f.write(f"{FILE_SHA}  questions.json\n")
    f.write(f"# questions-only fingerprint (identical on every rerun): {CONTENT_SHA}\n")
    f.write(f"# created {meta['created']}\n")

print(f"Saved {QUESTIONS_JSON}  ({os.path.getsize(QUESTIONS_JSON) / 1024**2:.1f} MB, {len(QUESTIONS):,} questions)")
print(f"\nSHA-256 of the file:           {FILE_SHA}")
print(f"Questions-only fingerprint:    {CONTENT_SHA}")
print(f"Saved {SHA_TXT}\n")
print(pd.DataFrame(counts).T.to_string())
REPORT["3.6_freeze"] = {"file_sha256": FILE_SHA, "questions_sha256": CONTENT_SHA, "created": meta["created"],
                        "counts": counts}

### Step 3.7: Show 10
- **What it does:** picks 10 questions at random (seed 0): 6 words and 4 photo questions, and prints each as a small table: the query, whether it has an answer, and the right pieces with their type and colour. Then it draws a picture for 2 photo questions that have an answer: the query photo on the left, then its 30 wardrobe pieces as thumbnails, with the right pieces framed in **green**. Saves the picture as `phase3_examples.png`.
- **Why:** rules checked by machine can still produce questions that make no sense to a person. Reading 10 and looking at two wardrobes is your own judgement of the exam, which the plan asks for.
- **Effect:** writes `phase3_examples.png` (about 1–2 MB). Takes about 10–20 seconds (it opens 62 pictures).
- **Predict:** in a photo question, will the green-framed pieces look like the query photo?
- **You should see:** 10 short blocks, each with its query ("black jeans", or a photo item), "has an answer: yes/no", and the right pieces. Then a figure with two rows: each has a big query photo and 30 thumbnails, 1 to 3 of them framed in green, all of the same colour and type as the query. Other pieces of the same type but another colour (look-alikes) may be there unframed: that's intended.
- **If it looks wrong:** a green piece that clearly doesn't match the query (a red bag for "black jeans") → note its qid; it would contradict 3.5. A query photo that also appears among the 30 thumbnails → leakage; bring it to me. Judge each of the 10 in the journal: does it make sense to you?
- **Teacher angle:** "Did you look at your questions?" Yes: 10 random questions, read one by one, and two wardrobes drawn with the right answers framed.

In [ ]:
# ---- Step 3.7: show 10 random questions, and draw 2 photo wardrobes ----
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED)
words_all = [q for q in QUESTIONS if q["track"] == "words"]
photo_all = [q for q in QUESTIONS if q["track"] == "photo"]
shown = ([words_all[i] for i in sorted(rng.choice(len(words_all), 6, replace=False))] +
         [photo_all[i] for i in sorted(rng.choice(len(photo_all), 4, replace=False))])
INFO = {s: ITEMS[s].set_index("item_id") for s in SPLITS}

for q in shown:
    info = INFO[q["split"]]
    if q["track"] == "words":
        head = f"query '{q['query_text']}' (form: {q['form']})"
    else:
        r = info.loc[q["query_item_id"]]
        head = f"query photo {q['query_item_id']} ({r['colour']} {r['type']})"
    print(f"{q['qid']} · {head} · has an answer: {'no' if q['absent'] else 'yes'}")
    if q["right"]:
        right_tbl = info.loc[q["right"], ["type", "colour"]].reset_index().to_string(index=False, header=False)
        print("\n".join("  right: " + line.strip() for line in right_tbl.splitlines()))
    else:
        print("  no right piece in this wardrobe: the honest answer is 'no sure match'")
    print()

def thumb(path, px=200):
    with Image.open(path) as im:
        im.draft("RGB", (px, px))
        t = im.convert("RGB")
    t.thumbnail((px, px))
    return t

draw = [q for q in shown if q["track"] == "photo" and not q["absent"]]
draw += [q for q in photo_all if not q["absent"] and q not in draw]
draw = draw[:2]
fig = plt.figure(figsize=(18, 13))
gs = fig.add_gridspec(7, 11, height_ratios=[1, 1, 1, 0.25, 1, 1, 1], hspace=0.6, wspace=0.15)
for n, q in enumerate(draw):
    top, info = n * 4, INFO[q["split"]]
    ax = fig.add_subplot(gs[top:top + 3, 0])
    r = info.loc[q["query_item_id"]]
    ax.imshow(thumb(r["image_path"], 400)); ax.set_xticks([]); ax.set_yticks([])
    for side in ax.spines.values():
        side.set_edgecolor("#1f77b4"); side.set_linewidth(3)
    ax.set_xlabel(f"QUERY {q['qid']}\n{r['colour']} {r['type']}", fontsize=9)
    for j, item in enumerate(q["wardrobe"]):
        ax = fig.add_subplot(gs[top + j // 10, 1 + j % 10])
        w = info.loc[item]
        ax.imshow(thumb(w["image_path"])); ax.set_xticks([]); ax.set_yticks([])
        is_right = item in q["right"]
        for side in ax.spines.values():
            side.set_edgecolor("#2ca02c" if is_right else "#bbbbbb"); side.set_linewidth(4 if is_right else 0.8)
        ax.set_xlabel(f"{w['type']}\n{w['colour']}", fontsize=6, color="#2ca02c" if is_right else "black")
fig.suptitle("Two photo questions: query (blue frame) and its 30-piece wardrobe; right pieces framed in green",
             fontsize=13)
EXAMPLES_PNG = os.path.join(OUT_DIR, "phase3_examples.png")
fig.savefig(EXAMPLES_PNG, dpi=80, bbox_inches="tight")
plt.show()
print(f"Saved {EXAMPLES_PNG}  (questions {', '.join(q['qid'] for q in draw)})")

### Step 3.8: Summary (bring me)
- **What it does:** prints, in one block, everything this phase asks you to bring back: counts per track and split, answer vs no answer, the leakage and integrity results, the left-out types and how many pairs were left out, both fingerprints, and the picture's path. Also saves `phase3_report.json` with every count.
- **Why:** one place to copy from into your journal and into the guide chat.
- **Effect:** writes `phase3_report.json` (a few KB).
- **Predict:** will every check line say PASS?
- **You should see:** a block titled `BRING ME (Phase 3)` with six numbered parts and the reminder to save a version for Phase 4.
- **If it looks wrong:** `Run steps 3.0 to 3.7 first` → a cell was skipped or the session restarted: rerun from 3.0 (everything is quick and seeded, so you get the same questions).
- **Teacher angle:** "Summarise how you built the exam in one minute." This block is that minute.

In [ ]:
# ---- Step 3.8: summary of everything to bring back ----
try:
    _ = (words_stats, photo_stats, integrity, self_in, group_in, FILE_SHA, CONTENT_SHA, EXAMPLES_PNG)
except NameError:
    raise RuntimeError("Run steps 3.0 to 3.7 first (Kaggle forgets everything when the session restarts).") from None

REPORT_JSON = os.path.join(OUT_DIR, "phase3_report.json")
REPORT["settings"] = {"seed": SEED, "versions": VERSIONS, "items_csv": ITEMS_CSV, "images_dir": IMAGES_DIR}
with open(REPORT_JSON, "w") as f:
    json.dump(REPORT, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

ok = lambda b: "PASS" if b else "FAIL"
print("=" * 72); print("BRING ME (Phase 3)"); print("=" * 72)
print("1. Questions per track and split (answer / no answer)")
for track, stats in [("words", words_stats), ("photo", photo_stats)]:
    for s in SPLITS:
        st = stats[s]
        print(f"   {track:<5} {s:<4} {st['questions']:,} · with answer {st['with_answer']} · "
              f"no answer {st['no_answer']} ({st['no_answer'] / st['questions']:.1%}) · k1/k2/k3 {st['k1']}/{st['k2']}/{st['k3']}")
print("\n2. Words forms: " + " · ".join(f"{s} type {words_stats[s]['type_form']} / colour+type "
                                      f"{words_stats[s]['colour_type_form']}" for s in SPLITS))
print(f"\n3. Leakage (photo): query in wardrobe {self_in} {ok(self_in == 0)} · "
      f"query group in wardrobe {group_in} {ok(group_in == 0)}")
print("\n4. Integrity (all 4,000 questions)")
for name, fails in integrity.items():
    print(f"   {name:<46} {fails}  {ok(fails == 0)}")
print("\n5. Left out (fewer than 3 items in the split)")
for s in SPLITS:
    lt = REPORT["3.1_types_left_out"][s]
    v = REPORT["3.1_vocabulary"][s]
    print(f"   {s:<4} types: " + (", ".join(f"{t} ({n})" for t, n in lt.items()) or "none") +
          f" · pairs left out: {v['pairs_total'] - v['pairs_allowed']} of {v['pairs_total']}")
print(f"\n6. Fingerprints\n   questions.json SHA-256:     {FILE_SHA}\n   questions-only fingerprint: {CONTENT_SHA}")
print(f"\nPicture: {EXAMPLES_PNG}\nReport:  {REPORT_JSON}")
print("\nReminder: click Save Version > 'Save & Run All (Commit)' so Phase 4 can attach this output.")
print("The background run writes a NEW questions.json (new creation time, so a new file SHA-256), but the")
print("questions-only fingerprint must be the SAME as above: check it in the saved version's output.")
print("Phase 4 uses the saved version's questions.json; download it, questions.sha256.txt, the report")
print("and the PNG from that version into bench/results/.")